# Calçada Segura · Assistente de normas de calçada com LLM local

**GS FIAP 2026.2 · Generative AI for Engineering · Engenharia de Software 4ES**

Tarefa: responder perguntas de cidadãos sobre as regras de calçada de São Paulo e de acessibilidade,
**citando a norma e o trecho**, e **recusar** quando a resposta não está na base.

Hipótese testada: o mesmo modelo local (Qwen2.5-1.5B-Instruct) **acerta mais e alucina menos** com contexto (RAG)
e com especialização (LoRA) do que sem contexto.

| Item da nota | Seção |
| --- | --- |
| 01 Tarefa e casos de teste com gabarito e "não consta" | 1 |
| 02 Base em trechos com fonte | 2 |
| 03 Busca com recall@1, recall@3 e limiar | 3 |
| 04 Geração fundamentada: JSON, citação e recusa | 4 |
| 05 Especialização LoRA x RAG simples | 5 |
| 06 Placar das três configurações | 6 |
| 07 Cinco erros reais | 7 |

**Antes de rodar:** Ambiente de execução → Alterar o tipo de ambiente de execução → **T4 GPU**. Depois, Ambiente de execução → Executar tudo.

In [ ]:
# 0 · Preparação: baixa o repositório e instala as dependências
import os
REPO = "https://github.com/Jpvalebm607/calcada-segura-ia.git"  # troque se o repositório tiver outro endereço
if not os.path.exists("/content/calcada-segura-ia"):
    !git clone -q {REPO} /content/calcada-segura-ia
%cd /content/calcada-segura-ia
!pip install -q -r requirements.txt

import json, sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
sys.path.insert(0, ".")
pd.set_option("display.max_colwidth", 120)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SEM GPU (troque para T4 antes de seguir)")

## 1 · Tarefa e casos de teste

38 perguntas escritas pela equipe: **28 que constam** na base (com resposta esperada, normas aceitas e termos que o trecho certo contém)
e **10 que não constam** (outras cidades, valores de mercado, Código de Trânsito, NBR 9050 paga etc.), para medir a recusa.

In [ ]:
from ia.avaliacao import carregar_casos
casos = carregar_casos(Path("avaliacao/casos_teste.json"))
df_casos = pd.DataFrame(casos)
print(df_casos.tipo.value_counts().to_string())
df_casos[["id", "tipo", "pergunta", "resposta_esperada", "fontes_aceitas"]]

## 2 · Base de conhecimento

Quatro normas oficiais, baixadas da fonte e quebradas **por artigo** (artigos longos viram partes). Cada trecho guarda fonte, artigo e URL.
O texto revogado (riscado no site do Planalto) fica de fora. A NBR 9050 **não** entra: é norma paga da ABNT.

In [ ]:
from ia.base import montar_base, FONTES
try:
    trechos = montar_base(Path("base"))
except Exception as e:
    print("Falha ao baixar alguma fonte:", e)
    print("Baixe manualmente as URLs abaixo, salve em base/fontes/ com o nome <ID>.<pdf|html> e rode de novo:")
    for f in FONTES: print(f"  {f['id']}.{f['formato']}  <-  {f['url']}")
    raise
tabela_base = pd.read_csv("base/tabela_base.csv")
print("Total de trechos:", len(trechos))
tabela_base

In [ ]:
# Exemplos de trechos
pd.DataFrame(trechos).sample(5, random_state=1)[["id", "artigo", "texto"]]

In [ ]:
# Conferência do gabarito: cada pergunta que consta precisa ter ao menos 1 trecho relevante na base
from ia.avaliacao import relevantes
sem_relevante = [c["id"] for c in casos if c["tipo"] == "consta" and not relevantes(c, trechos)]
for c in casos:
    if c["tipo"] == "consta":
        print(c["id"], "->", len(relevantes(c, trechos)), "trecho(s) relevante(s)")
print("\nCasos SEM trecho relevante (revisar trecho_contem):", sem_relevante or "nenhum")

## 3 · Busca: recall@1, recall@3 e limiar

Embeddings `intfloat/multilingual-e5-base`, similaridade de cosseno. **recall@k** = % das perguntas que constam cujo trecho relevante
aparece entre os k primeiros. O **limiar** decide quando recusar sem nem chamar o modelo: é o valor de similaridade que melhor separa
perguntas que constam das que não constam (maior acurácia balanceada).

In [ ]:
from ia.busca import Buscador
from ia.avaliacao import avaliar_busca, escolher_limiar
buscador = Buscador(trechos)
buscador.salvar(Path("base"))
df_busca = avaliar_busca(casos, trechos, buscador)
avaliaveis = df_busca[df_busca.tipo == "consta"].dropna(subset=["acerto@1"])
recall1 = avaliaveis["acerto@1"].astype(bool).mean()
recall3 = avaliaveis["acerto@3"].astype(bool).mean()
print(f"recall@1 = {recall1:.1%}   recall@3 = {recall3:.1%}   (sobre {len(avaliaveis)} perguntas que constam)")
LIMIAR, curva = escolher_limiar(df_busca)
print(f"Limiar escolhido: {LIMIAR:.4f}")
Path("avaliacao").mkdir(exist_ok=True)
df_busca.to_csv("avaliacao/busca.csv", index=False)
json.dump({"limiar": LIMIAR, "recall@1": recall1, "recall@3": recall3}, open("avaliacao/limiar.json", "w"), indent=1)
df_busca

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for tipo, cor in [("consta", "tab:green"), ("nao_consta", "tab:red")]:
    ax[0].hist(df_busca[df_busca.tipo == tipo].score_top1, bins=15, alpha=0.6, label=tipo, color=cor)
ax[0].axvline(LIMIAR, color="k", ls="--", label=f"limiar {LIMIAR:.3f}")
ax[0].set_title("Similaridade do 1º trecho"); ax[0].set_xlabel("cosseno"); ax[0].legend()
ax[1].plot(curva.limiar, curva.consta_aceitas, label="consta aceitas")
ax[1].plot(curva.limiar, curva.nao_consta_recusadas, label="não consta recusadas")
ax[1].plot(curva.limiar, curva.balanceada, label="acurácia balanceada", lw=2)
ax[1].axvline(LIMIAR, color="k", ls="--"); ax[1].set_title("Escolha do limiar"); ax[1].legend()
plt.tight_layout(); plt.savefig("avaliacao/limiar.png", dpi=120); plt.show()

## 4 · Geração fundamentada

Modelo local **Qwen2.5-1.5B-Instruct** (roda na T4 do Colab). A saída é sempre um JSON validado com Pydantic:
`resposta`, `fonte`, `trecho_id`, `nao_consta`. Se o 1º trecho fica abaixo do limiar, a resposta é a recusa "Não consta na base".

Configurações comparadas:
- **A · sem contexto**: o mesmo modelo, só com a pergunta
- **B · RAG simples**: 3 trechos mais parecidos + limiar
- **C · RAG + LoRA**: igual à B, com o modelo especializado (seção 5)

In [ ]:
from ia.gerador import Gerador, responder
from ia.avaliacao import pontuar
gerador = Gerador()
exemplo = responder(casos[0]["pergunta"], "B_rag_simples", gerador, buscador, LIMIAR)
print("Pergunta:", casos[0]["pergunta"])
print(json.dumps(exemplo["saida"], ensure_ascii=False, indent=1))
print("Trechos recuperados:", exemplo["recuperados"])

In [ ]:
resultados = []
def rodar(config):
    for c in casos:
        r = responder(c["pergunta"], config, gerador, buscador, LIMIAR)
        resultados.append({"config": config, "id": c["id"], "tipo": c["tipo"], "pergunta": c["pergunta"],
                           "esperado": c["resposta_esperada"], "saida": json.dumps(r["saida"], ensure_ascii=False),
                           "bruto": r["bruto"], "recuperados": r["recuperados"], "tempo_s": r["tempo_s"],
                           "relevantes": sorted(relevantes(c, trechos)) if c["tipo"] == "consta" else [],
                           **pontuar(c, r)})
    print(config, "concluída")

rodar("A_sem_contexto")
rodar("B_rag_simples")

## 5 · Especialização com LoRA

**Dados de treino sintéticos, sem usar as perguntas de teste:**
- *positivos*: o modelo-base lê um trecho e escreve uma pergunta de cidadão e a resposta curta; só fica o par cuja resposta está apoiada no trecho;
- *negativos*: perguntas fora do tema com trechos aleatórios da base, ensinando a recusar.

Treino LoRA (r=16, q/k/v/o) por 2 épocas. O adaptador fica salvo em `modelo/lora_calcada`.

In [ ]:
from ia import lora
t0 = time.time()
pares = lora.gerar_pares(gerador, trechos, n=200)
exemplos = lora.montar_exemplos(pares, trechos, buscador)
print(f"Pares positivos aprovados: {len(pares)} | exemplos de treino (com negativos): {len(exemplos)} | {time.time()-t0:.0f}s")
with open("avaliacao/treino_lora.jsonl", "w", encoding="utf-8") as f:
    for e in exemplos: f.write(json.dumps(e, ensure_ascii=False) + "\n")
pd.DataFrame([{"pergunta": p["pergunta"], "resposta": p["resposta"], "trecho": p["trecho"]["id"]} for p in pares]).head(8)

In [ ]:
historico = lora.treinar(gerador, exemplos, "modelo/lora_calcada", epocas=2)
perdas = [(h["step"], h["loss"]) for h in historico if "loss" in h]
plt.plot(*zip(*perdas)); plt.title("Perda do treino LoRA"); plt.xlabel("passo"); plt.ylabel("loss")
plt.savefig("avaliacao/perda_lora.png", dpi=120); plt.show()

In [ ]:
rodar("C_rag_lora")

## 6 · Placar das três configurações

In [ ]:
from ia.avaliacao import placar
df = pd.DataFrame(resultados)
df.to_csv("avaliacao/resultados.csv", index=False)
tab = placar(df)
tab.to_csv("avaliacao/placar.csv")
tab

In [ ]:
tab[["acuracia_%", "alucinacao_%", "recusa_correta_%", "json_valido_%"]].plot.bar(figsize=(10, 4), rot=0)
plt.title("Placar por configuração"); plt.ylabel("%"); plt.ylim(0, 105); plt.legend(loc="upper left", ncol=4)
plt.tight_layout(); plt.savefig("avaliacao/placar.png", dpi=120); plt.show()

**Interpretação** *(escrever depois de rodar, com os números do placar)*: compare A x B (efeito do contexto) e B x C (efeito do LoRA)
em acurácia, alucinação e recusa correta, e comente o custo em tempo.

## 7 · Erros reais

Todas as respostas erradas das configurações B e C, com o que foi recuperado. Escolha 5 para o relatório:
pergunta, resposta errada, causa (busca trouxe o trecho errado? modelo ignorou o trecho? formato inválido? recusa indevida?) e correção proposta.

In [ ]:
erros = df[(df.config.isin(["B_rag_simples", "C_rag_lora"])) & (~df.correto)]
erros[["config", "id", "pergunta", "esperado", "saida", "recuperados", "relevantes"]]

In [ ]:
# 8 · Salvar e baixar os resultados para subir no GitHub
!zip -qr /content/resultados_ia.zip base/trechos.jsonl base/tabela_base.csv base/embeddings.npy base/fontes avaliacao modelo/lora_calcada
from google.colab import files
files.download("/content/resultados_ia.zip")
print("Depois: Arquivo → Baixar → Baixar o .ipynb (o notebook com as saídas) e suba em notebooks/")